# Appendix A1: NeuroGym Cognitive Tasks Survey
## From Puzzle Boxes to Gymnasium Environments

**Spinning Up in Active Inference** | Appendix

---

Before we can train neural networks on cognitive tasks or build Active Inference agents for them, we need to understand the tasks themselves. This notebook surveys the classical neuroscience paradigms that NeuroGym implements as Gymnasium-compatible environments, tracing each task from its origins in animal behavior to its modern computational form.

By the end of this notebook you will:
1. Understand the three waves of cognitive task implementation (physical → software → environment libraries)
2. Be able to use the NeuroGym Gymnasium API to run and visualize six classical paradigms
3. See how each task maps onto the RL↔AIF framework from the core curriculum
4. Have a taxonomy for organizing tasks by cognitive demands

**Prerequisites:** Module 1 (behavioral background), basic Python.  
**Time:** ~60 minutes.

## A1.1 Three Waves of Cognitive Task Implementation

The history of cognitive tasks in neuroscience mirrors the broader trajectory of experimental science: from physical apparatus, to software simulations, to standardized open-source libraries.

### Wave 1: Physical Apparatus (1898–1980s)

Thorndike's puzzle boxes (1898), Skinner's operant conditioning chambers (1930s), and the Wisconsin Card Sorting Test (Berg, 1948) all required purpose-built physical equipment. Each lab constructed its own apparatus, making replication difficult and computational modeling nearly impossible. The animal *was* the agent, and the box *was* the environment.

### Wave 2: Computerized Tasks (1990s–2010s)

As neuroscience moved from freely-behaving animals to head-fixed monkeys and human fMRI, tasks migrated to computer screens. Software like PsychToolbox (MATLAB) and PsychoPy (Python) let researchers present stimuli and record responses digitally. Training RNNs on these tasks became possible with packages like **PsychRNN** (Ehrlich et al., eNeuro 2021), but these were tightly coupled to specific deep learning frameworks — PsychRNN required TensorFlow 1.x, which is now effectively dead.

### Wave 3: Environment Libraries (2019–present)

**NeuroGym** (Yang & Wang, 2020) took a different approach: implement tasks as Gymnasium-compatible environments with a standard `reset()`/`step()` API. This decouples the *task* from the *training algorithm*, letting researchers use any framework — PyTorch, JAX, or even hand-coded Active Inference agents.

| Feature | PsychRNN | NeuroGym |
|---------|----------|----------|
| Framework | TensorFlow 1.x only | Any (Gymnasium API) |
| Tasks | ~10 | 30+ |
| Training | Built-in RNN training | BYO training loop |
| Status | Archived | Active |

This shift matters for us because Active Inference agents are not standard deep RL agents — they need a generic environment interface, not a framework-specific training harness. NeuroGym gives us exactly that.

In [ ]:
# ── Setup ───────────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import sys; sys.path.insert(0, '..')
from utils.plotting import COLORS, _setup_style, dual_perspective_box, plot_matrix_heatmap
_setup_style()

np.random.seed(42)

# ── NeuroGym import with fallback ─────────────────────────────────────────────────────
try:
    import neurogym as ngym
    NEUROGYM_AVAILABLE = True
    print(f"NeuroGym {ngym.__version__} loaded — {len(ngym.all_envs())} tasks available")
except ImportError:
    NEUROGYM_AVAILABLE = False
    print("NeuroGym not installed — using synthetic demonstrations.")
    print("Install with: pip install neurogym")

In [ ]:
# ── Helper: Trial Timecourse Visualization ───────────────────────────────────────

def plot_task_timecourse(observations, actions, rewards, title, obs_labels=None):
    """Plot a single trial's timecourse: observations, actions, rewards."""
    fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
    t = np.arange(len(observations))

    # Observations
    if observations.ndim == 2:
        for ch in range(observations.shape[1]):
            label = obs_labels[ch] if obs_labels and ch < len(obs_labels) else f'ch{ch}'
            axes[0].plot(t, observations[:, ch], linewidth=1.5, label=label)
        axes[0].legend(fontsize=8, loc='upper right')
    else:
        axes[0].plot(t, observations, color=COLORS['neutral'], linewidth=1.5)
    axes[0].set_ylabel('Observation')
    axes[0].set_title(title)

    # Actions
    axes[1].step(t[:len(actions)], actions, color=COLORS['rl'], linewidth=1.5, where='mid')
    axes[1].set_ylabel('Action')

    # Rewards
    reward_colors = [COLORS['reward'] if r > 0 else COLORS['danger'] if r < 0 else COLORS['neutral'] for r in rewards]
    axes[2].bar(t[:len(rewards)], rewards, color=reward_colors, alpha=0.7)
    axes[2].set_ylabel('Reward')
    axes[2].set_xlabel('Timestep')

    plt.tight_layout()
    plt.show()

## A1.2 The Gymnasium API

Every NeuroGym task follows the Gymnasium interface:

```python
env = ngym.make('TaskName-v0')     # Create environment
ob = env.reset()                    # Get initial observation
ob, reward, done, info = env.step(action)  # Take one step
```

This is the same API used by Atari, MuJoCo, and every other Gymnasium environment. The key difference for cognitive tasks is that **observations are continuous vectors** (not pixel images) and **trials have internal temporal structure** (fixation → stimulus → delay → response).

The `info` dictionary often contains ground-truth labels (e.g., which stimulus was correct) that are invisible to the agent but useful for analysis.

In [ ]:
# ── Gymnasium API Demo ───────────────────────────────────────────────────────────
if NEUROGYM_AVAILABLE:
    env = ngym.make('GoNogo-v0')
    ob = env.reset()
    print(f"Observation shape: {ob.shape}")
    print(f"Action space: {env.action_space}")
    print(f"Observation space: {env.observation_space}")

    # Run a few steps
    for step in range(5):
        action = env.action_space.sample()
        ob, reward, done, info = env.step(action)
        print(f"  Step {step}: action={action}, reward={reward:.1f}, done={done}")
        if done:
            ob = env.reset()
else:
    print("Gymnasium API pattern:")
    print("  ob = env.reset()                    # Initial observation")
    print("  ob, reward, done, info = env.step(a) # Take action, get feedback")
    print()
    print("Typical observation vector for a cognitive task:")
    print(f"  fixation input:  1.0  (maintain gaze)")
    print(f"  stimulus ch. 1:  0.0  (no stimulus yet)")
    print(f"  stimulus ch. 2:  0.0  (no stimulus yet)")
    print()
    print("Actions: 0=fixate, 1=respond_left, 2=respond_right (task-dependent)")

## A1.3 Six Classical Paradigms

We now survey six tasks that span the major cognitive demands: perception, working memory, decision-making, motor control, and learning. For each task, we provide:

1. **Historical context** — where the task came from and what it was designed to measure
2. **Trial visualization** — what observations, actions, and rewards look like over time
3. **RL ↔ AIF comparison** — how each framework approaches the task

If NeuroGym is installed, we run the real environments. Otherwise, synthetic data demonstrates the same structure.

### Paradigm 1: Go/No-Go

**Origin:** Rooted in Thorndike's operant conditioning (1898) and Skinner's reinforcement schedules (1938). The modern Go/No-Go task isolates *response inhibition*: the ability to withhold a prepotent response when cued.

**Task structure:**
```
  Fixation      Stimulus       Response
 [________]  [==GO=======]  [→ respond!]   → reward
 [________]  [--nogo-----]  [→ withhold!]  → reward
 [________]  [--nogo-----]  [→ respond ]   → penalty
```

**Neuroscience significance:** Damage to the right inferior frontal gyrus impairs No-Go performance, linking this task to prefrontal inhibitory control. In ADHD research, Go/No-Go commission errors (responding on No-Go trials) are a core biomarker.

**NeuroGym implementation:** `GoNogo-v0` presents a fixation period, then a stimulus. Go stimuli are strong signals; No-Go stimuli are weak. The agent must respond (action=1) to Go trials and withhold (action=0) on No-Go trials.

In [ ]:
# ── Paradigm 1: Go/No-Go ───────────────────────────────────────────────────────

if NEUROGYM_AVAILABLE:
    env = ngym.make('GoNogo-v0')
    obs_list, act_list, rew_list = [], [], []
    ob = env.reset()
    for _ in range(100):
        action = env.action_space.sample()
        ob, reward, done, info = env.step(action)
        obs_list.append(ob)
        act_list.append(action)
        rew_list.append(reward)
        if done:
            break
    observations = np.array(obs_list)
    actions = np.array(act_list)
    rewards = np.array(rew_list)
else:
    # Synthetic Go/No-Go trial data
    n_steps = 60
    observations = np.zeros((n_steps, 3))  # fixation, stimulus_go, stimulus_nogo
    actions = np.zeros(n_steps, dtype=int)
    rewards = np.zeros(n_steps)

    # Trial 1: Go trial (steps 0-29)
    observations[0:10, 0] = 1.0                      # fixation
    observations[10:25, 1] = 0.8 + 0.1 * np.random.randn(15)  # go stimulus (strong)
    observations[10:25, 0] = 1.0
    actions[25:30] = 1                                # respond
    rewards[25] = 1.0                                 # correct!

    # Trial 2: No-Go trial (steps 30-59)
    observations[30:40, 0] = 1.0                      # fixation
    observations[40:55, 2] = 0.2 + 0.1 * np.random.randn(15)  # nogo stimulus (weak)
    observations[40:55, 0] = 1.0
    actions[55:60] = 0                                # withhold
    rewards[55] = 1.0                                 # correct!

plot_task_timecourse(observations, actions, rewards,
                     'Go/No-Go Task: Two Trials',
                     obs_labels=['Fixation', 'Go stimulus', 'NoGo stimulus'])

In [ ]:
# ── Go/No-Go: RL vs AIF ────────────────────────────────────────────────────────
dual_perspective_box(
    rl_text="Learn a stimulus→response mapping via reward prediction errors. "
            "The agent learns Q(s,a) values: Q(go_stim, respond) > Q(go_stim, withhold). "
            "Response inhibition is just learning that Q(nogo_stim, withhold) > Q(nogo_stim, respond). "
            "No explicit model of trial types is needed.",
    aif_text="Infer the hidden trial type (go vs. no-go) from observations, then select actions "
            "that minimize expected free energy. The A-matrix encodes P(obs|state): strong stimuli "
            "are evidence for 'go' trials, weak stimuli for 'no-go'. The C-vector (preferences) "
            "encodes that reward is preferred. Inhibition emerges from accurate state inference, "
            "not from suppressing a learned response.",
    title="Go/No-Go: Learned Inhibition vs. Inferred Trial Type"
)

### Paradigm 2: N-Armed Bandit

**Origin:** The multi-armed bandit problem was formalized by Robbins (1952) but echoes the earliest choice experiments in animal behavior. A pigeon pecking one of several keys, a rat choosing between levers — these are all bandit problems. The name comes from imagining a gambler facing a row of slot machines ("one-armed bandits"), not knowing which has the best payout.

**Task structure:**
```
  Choose arm     Observe reward
  [arm 0] -----> r ~ Bernoulli(p=0.8)  ← best arm
  [arm 1] -----> r ~ Bernoulli(p=0.2)
  [arm 2] -----> r ~ Bernoulli(p=0.5)
```

**Neuroscience significance:** The bandit task is the canonical exploration/exploitation paradigm. Dopaminergic neurons in the ventral tegmental area (VTA) track reward prediction errors that are mathematically equivalent to the TD error in RL. In human fMRI, the rostrolateral prefrontal cortex tracks the "relative uncertainty" of unchosen options — a signal closely related to epistemic value in Active Inference.

**NeuroGym implementation:** `Bandit-v0` implements a multi-armed bandit where reward probabilities may drift over time, requiring ongoing exploration.

In [ ]:
# ── Paradigm 2: N-Armed Bandit ─────────────────────────────────────────────────

n_trials = 200
n_arms = 3
arm_probs = [0.8, 0.2, 0.5]  # True reward probabilities

if NEUROGYM_AVAILABLE:
    try:
        env = ngym.make('Bandit-v0')
        obs_list, act_list, rew_list = [], [], []
        ob = env.reset()
        for _ in range(n_trials):
            action = env.action_space.sample()
            ob, reward, done, info = env.step(action)
            obs_list.append(ob)
            act_list.append(action)
            rew_list.append(reward)
            if done:
                ob = env.reset()
        bandit_actions = np.array(act_list)
        bandit_rewards = np.array(rew_list)
    except Exception:
        NEUROGYM_AVAILABLE = False  # Fall through to synthetic

if not NEUROGYM_AVAILABLE:
    # Synthetic bandit data: random agent
    bandit_actions = np.random.randint(0, n_arms, n_trials)
    bandit_rewards = np.array([np.random.binomial(1, arm_probs[a]) for a in bandit_actions])

# Visualize: cumulative reward per arm
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: action choices over time
arm_colors = [COLORS['rl'], COLORS['danger'], COLORS['reward']]
for arm in range(n_arms):
    mask = bandit_actions == arm
    axes[0].scatter(np.where(mask)[0], bandit_actions[mask],
                    c=arm_colors[arm], alpha=0.4, s=10, label=f'Arm {arm} (p={arm_probs[arm]})')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Arm Chosen')
axes[0].set_title('Bandit: Action Choices (Random Agent)')
axes[0].legend(fontsize=9)

# Right: cumulative reward
cum_reward = np.cumsum(bandit_rewards)
axes[1].plot(cum_reward, color=COLORS['reward'], linewidth=2)
axes[1].fill_between(range(len(cum_reward)), cum_reward, alpha=0.2, color=COLORS['reward'])
# Optimal agent line
optimal_rate = max(arm_probs)
axes[1].plot([0, n_trials], [0, n_trials * optimal_rate],
             '--', color=COLORS['highlight'], linewidth=2, label=f'Optimal ({optimal_rate})')
axes[1].set_xlabel('Trial')
axes[1].set_ylabel('Cumulative Reward')
axes[1].set_title('Bandit: Cumulative Reward vs Optimal')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Bandit: RL vs AIF ──────────────────────────────────────────────────────────
dual_perspective_box(
    rl_text="Epsilon-greedy: exploit the best-known arm most of the time, but randomly explore with "
            "probability epsilon. UCB (Upper Confidence Bound) adds an exploration bonus proportional "
            "to uncertainty. Thompson Sampling samples from posterior reward distributions. "
            "Exploration is a *heuristic* added on top of value maximization.",
    aif_text="Epistemic value (information gain about hidden arm probabilities) is part of the expected "
            "free energy objective. Uncertain arms have high epistemic value, so the agent explores them "
            "to reduce uncertainty about the generative model. Exploration is not a heuristic — it "
            "emerges naturally from minimizing uncertainty. As beliefs sharpen, epistemic value drops "
            "and pragmatic (reward-seeking) value dominates: exploitation follows automatically.",
    title="Bandits: Heuristic Exploration vs. Principled Information-Seeking"
)

### Paradigm 3: Daw Two-Step Task

**Origin:** Designed by Daw et al. (2011, Neuron) specifically to dissociate model-free from model-based learning in humans. This is the most important task in the model-based RL literature and connects directly to Module 5 of the core curriculum.

**Task structure:**
```
  Stage 1              Stage 2              Outcome
  Choose A or B  -->  State X (70%) or    -->  Reward?
                      State Y (30%)            (drifting probabilities)
  
  A ── 70% ──> X ──> reward p(t)
    └─ 30% ──> Y ──> reward q(t)
  B ── 70% ──> Y ──> reward q(t)
    └─ 30% ──> X ──> reward p(t)
```

**Neuroscience significance:** The key diagnostic is what happens after a *rare* transition. If a model-free agent gets reward after a rare transition (A→30%→Y→reward), it increases P(A) because A led to reward. A model-based agent *decreases* P(A) because it knows A usually leads to X, not Y — so if Y was rewarding, it should choose B next time (which usually leads to Y). This "stay/switch" analysis is the gold standard for detecting model-based planning in behavior.

**NeuroGym implementation:** `DawTwoStep-v0` implements the full two-step structure with drifting reward probabilities.

In [ ]:
# ── Paradigm 3: Daw Two-Step Task ──────────────────────────────────────────────

n_trials = 150

if NEUROGYM_AVAILABLE:
    try:
        env = ngym.make('DawTwoStep-v0')
        obs_list, act_list, rew_list = [], [], []
        ob = env.reset()
        for _ in range(n_trials * 3):  # Multiple steps per trial
            action = env.action_space.sample()
            ob, reward, done, info = env.step(action)
            obs_list.append(ob)
            act_list.append(action)
            rew_list.append(reward)
            if done:
                ob = env.reset()
        ts_rewards = np.array(rew_list)
    except Exception:
        pass

# Synthetic two-step demonstration
# Simulate drifting reward probabilities
t = np.arange(n_trials)
p_X = 0.5 + 0.3 * np.sin(2 * np.pi * t / 80) + 0.05 * np.random.randn(n_trials)
p_Y = 0.5 - 0.3 * np.sin(2 * np.pi * t / 80) + 0.05 * np.random.randn(n_trials)
p_X = np.clip(p_X, 0.05, 0.95)
p_Y = np.clip(p_Y, 0.05, 0.95)

# Simulate random agent
stage1_choices = np.random.randint(0, 2, n_trials)  # 0=A, 1=B
transitions = np.array([np.random.choice([0, 1], p=[0.7, 0.3]) if c == 0
                        else np.random.choice([1, 0], p=[0.7, 0.3])
                        for c in stage1_choices])  # 0=X, 1=Y
is_common = ((stage1_choices == 0) & (transitions == 0)) | ((stage1_choices == 1) & (transitions == 1))
outcomes = np.array([np.random.binomial(1, p_X[i]) if transitions[i] == 0
                     else np.random.binomial(1, p_Y[i]) for i in range(n_trials)])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: drifting reward probabilities
axes[0].plot(t, p_X, color=COLORS['rl'], linewidth=2, label='P(reward | State X)')
axes[0].plot(t, p_Y, color=COLORS['aif'], linewidth=2, label='P(reward | State Y)')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Reward Probability')
axes[0].set_title('Two-Step: Drifting Reward Probabilities')
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 1)

# Right: stay probability by transition type and reward
stay = np.zeros(4)  # [common+rew, common+norew, rare+rew, rare+norew]
counts = np.zeros(4)
for i in range(n_trials - 1):
    stayed = stage1_choices[i+1] == stage1_choices[i]
    idx = (0 if is_common[i] else 2) + (0 if outcomes[i] == 1 else 1)
    stay[idx] += stayed
    counts[idx] += 1
stay_prob = stay / np.maximum(counts, 1)

x = [0, 1]
axes[1].bar([0, 1], stay_prob[:2], 0.35, label='Common', color=COLORS['rl'], alpha=0.8)
axes[1].bar([0.35, 1.35], stay_prob[2:], 0.35, label='Rare', color=COLORS['aif'], alpha=0.8)
axes[1].set_xticks([0.175, 1.175])
axes[1].set_xticklabels(['Rewarded', 'Unrewarded'])
axes[1].set_ylabel('P(stay)')
axes[1].set_title('Stay Probability (Random Agent)')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

print("Key diagnostic: A model-based agent shows a CROSSOVER interaction")
print("(higher stay after common+rewarded AND rare+unrewarded).")
print("A model-free agent shows a MAIN EFFECT of reward (higher stay after any reward).")

In [ ]:
# ── Two-Step: RL vs AIF ────────────────────────────────────────────────────────
dual_perspective_box(
    rl_text="Model-free RL (SARSA, Q-learning) ignores transition structure — it just updates values "
            "based on received reward. Model-based RL maintains a transition model T(s'|s,a) and "
            "uses it to plan (Dyna, MCTS). The two-step task was designed to dissociate these: "
            "the signature 'crossover interaction' in stay probabilities is the behavioral fingerprint "
            "of model-based planning.",
    aif_text="In Active Inference, the B-matrix IS the transition model — there is no model-free "
            "mode. B[action] encodes P(s'|s, action), directly representing the 70/30 transition "
            "structure. Planning via expected free energy naturally uses this structure, so AIF agents "
            "are model-based by construction. The two-step task is trivially handled because the "
            "agent always consults its generative model before acting.",
    title="Two-Step: Optional Model Use vs. Model-Based by Construction"
)

### Paradigm 4: Perceptual Decision Making (Random Dots)

**Origin:** The random dot motion (RDM) task, pioneered by Newsome and colleagues in the 1980s-90s, is the fruit fly of perceptual neuroscience. A cloud of dots moves on screen; some fraction moves coherently in one direction while the rest move randomly. The subject reports the net direction of motion.

**Task structure:**
```
  Fixation    Stimulus (noisy evidence)    Decision
 [________]  [...<-<-..<-...<-.<-....]  [→ left!]    coherence = 50%
 [________]  [..->...<-..>...<-..>...]  [→ ???]     coherence = 5%
```

**Neuroscience significance:** Recordings from area LIP in monkeys showed neurons that *accumulate evidence* over time, ramping up until crossing a threshold. This inspired the drift-diffusion model (DDM), where a decision variable integrates noisy evidence until hitting a bound. The speed-accuracy tradeoff — fast but error-prone vs. slow but accurate — is controlled by the bound height.

**NeuroGym implementation:** `PerceptualDecisionMaking-v0` provides a fixation period followed by noisy stimulus inputs, with coherence as a parameter.

In [ ]:
# ── Paradigm 4: Perceptual Decision Making ─────────────────────────────────────

if NEUROGYM_AVAILABLE:
    try:
        env = ngym.make('PerceptualDecisionMaking-v0')
        obs_list, act_list, rew_list = [], [], []
        ob = env.reset()
        for _ in range(120):
            action = env.action_space.sample()
            ob, reward, done, info = env.step(action)
            obs_list.append(ob)
            act_list.append(action)
            rew_list.append(reward)
            if done:
                break
        observations = np.array(obs_list)
        actions = np.array(act_list)
        rewards = np.array(rew_list)
        plot_task_timecourse(observations, actions, rewards,
                             'Perceptual Decision Making: Single Trial')
    except Exception:
        pass

# Synthetic PDM: evidence accumulation at different coherences
coherences = [0.05, 0.15, 0.30, 0.60]
n_steps = 80
true_direction = 1  # rightward

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: evidence accumulation traces
for coh in coherences:
    evidence = np.cumsum(coh * true_direction + np.random.randn(n_steps) * 0.3)
    axes[0].plot(evidence, linewidth=1.5, label=f'coh={coh:.0%}')

axes[0].axhline(y=3.0, color=COLORS['reward'], linestyle='--', linewidth=1, label='Right bound')
axes[0].axhline(y=-3.0, color=COLORS['danger'], linestyle='--', linewidth=1, label='Left bound')
axes[0].axhline(y=0, color='gray', linestyle=':', linewidth=0.5)
axes[0].set_xlabel('Timestep')
axes[0].set_ylabel('Accumulated Evidence')
axes[0].set_title('Evidence Accumulation (Drift-Diffusion)')
axes[0].legend(fontsize=8)

# Right: psychometric curve (synthetic)
signed_coherences = np.array([-0.6, -0.3, -0.15, -0.05, 0.05, 0.15, 0.3, 0.6])
# Logistic psychometric function
p_right = 1 / (1 + np.exp(-15 * signed_coherences))
# Add noise for realism
p_right_noisy = np.clip(p_right + 0.03 * np.random.randn(len(p_right)), 0, 1)

axes[1].plot(signed_coherences * 100, p_right, '-', color=COLORS['neutral'],
             linewidth=2, label='Logistic fit')
axes[1].scatter(signed_coherences * 100, p_right_noisy, color=COLORS['rl'],
                s=60, zorder=5, label='Simulated data')
axes[1].axhline(y=0.5, color='gray', linestyle=':', linewidth=0.5)
axes[1].axvline(x=0, color='gray', linestyle=':', linewidth=0.5)
axes[1].set_xlabel('Signed Coherence (%)')
axes[1].set_ylabel('P(choose right)')
axes[1].set_title('Psychometric Curve')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── PDM: RL vs AIF ─────────────────────────────────────────────────────────────
dual_perspective_box(
    rl_text="An RNN trained with policy gradient learns to integrate evidence over time and set "
            "an implicit decision threshold. The network's recurrent dynamics implement something "
            "like a drift-diffusion process, but this emerges from training rather than being "
            "built in. The speed-accuracy tradeoff is controlled by reward structure (penalty "
            "for time vs. penalty for errors).",
    aif_text="Sequential Bayesian belief updating IS evidence accumulation. At each timestep, the "
            "agent updates P(direction | observations_so_far) using the A-matrix likelihood. The "
            "drift-diffusion model emerges as a natural consequence of iterative Bayesian inference "
            "with Gaussian noise. The decision threshold corresponds to the precision of beliefs "
            "needed before expected free energy favors committing to an action over gathering "
            "more evidence.",
    title="PDM: Learned Integration vs. Bayesian Belief Updating"
)

### Paradigm 5: Delayed Match-to-Sample (Working Memory)

**Origin:** Working memory tasks date to Jacobsen's (1935) delayed-response experiments with monkeys. The delayed match-to-sample (DMS) variant presents a sample stimulus, imposes a delay, then presents test stimuli — the subject must identify which test matches the sample. Fuster & Alexander (1971) and Goldman-Rakic (1995) showed that neurons in prefrontal cortex fire persistently during the delay, forming the neural basis of working memory.

**Task structure:**
```
  Fixation    Sample     Delay (blank)    Test          Response
 [________]  [STIM A]  [____________]  [A or B?]  --> [match A!]  → reward
```

**Neuroscience significance:** The delay period is the crucial test. During this interval, no task-relevant stimulus is present, so the agent must maintain a representation purely from memory. In recurrent neural networks, this maps to sustained activity patterns (fixed points or slow dynamics in the network state).

**NeuroGym implementation:** `DelayMatchSample-v0` presents a sample stimulus, a variable-length delay, and then a test. The agent must respond based on whether the test matches the sample.

In [ ]:
# ── Paradigm 5: Delayed Match-to-Sample ────────────────────────────────────────

if NEUROGYM_AVAILABLE:
    try:
        env = ngym.make('DelayMatchSample-v0')
        obs_list, act_list, rew_list = [], [], []
        ob = env.reset()
        for _ in range(120):
            action = env.action_space.sample()
            ob, reward, done, info = env.step(action)
            obs_list.append(ob)
            act_list.append(action)
            rew_list.append(reward)
            if done:
                break
        observations = np.array(obs_list)
        actions = np.array(act_list)
        rewards = np.array(rew_list)
        plot_task_timecourse(observations, actions, rewards,
                             'Delayed Match-to-Sample: Single Trial')
    except Exception:
        pass

# Synthetic DMS trial
n_steps = 80
sample_id = 1  # Stimulus A

# Build observation array: [fixation, stim_channel_1, stim_channel_2]
obs_dms = np.zeros((n_steps, 3))
act_dms = np.zeros(n_steps, dtype=int)
rew_dms = np.zeros(n_steps)

# Fixation period: steps 0-14
obs_dms[0:15, 0] = 1.0

# Sample period: steps 15-29 (stimulus A = channel 1 active)
obs_dms[15:30, 0] = 1.0
obs_dms[15:30, 1] = 0.9 + 0.05 * np.random.randn(15)

# Delay period: steps 30-54 (only fixation, no stimulus)
obs_dms[30:55, 0] = 1.0

# Test period: steps 55-69 (show both stimuli, agent must identify match)
obs_dms[55:70, 0] = 0.0  # fixation off = response window
obs_dms[55:70, 1] = 0.85 + 0.05 * np.random.randn(15)  # Stimulus A (match)
obs_dms[55:70, 2] = 0.8 + 0.05 * np.random.randn(15)   # Stimulus B (non-match)

# Response: match = action 1
act_dms[65:70] = 1
rew_dms[65] = 1.0  # Correct match

plot_task_timecourse(obs_dms, act_dms, rew_dms,
                     'Delayed Match-to-Sample: Synthetic Trial',
                     obs_labels=['Fixation', 'Stim A', 'Stim B'])

# Annotate task phases
print("Task phases: Fixation (0-14) | Sample (15-29) | Delay (30-54) | Test (55-69) | Response (65+)")
print("The delay period is the critical test of working memory.")
print("No task-relevant stimulus is present — the agent must remember the sample.")

In [ ]:
# ── DMS: RL vs AIF ─────────────────────────────────────────────────────────────
dual_perspective_box(
    rl_text="Recurrent neural networks (LSTMs, GRUs) trained with RL maintain information in their "
            "hidden state during the delay. Memory is an emergent property of the recurrent dynamics "
            "— the network learns to create stable activity patterns that encode the sample identity. "
            "There is no explicit 'memory' module; it is distributed across network weights.",
    aif_text="The belief state P(sample_identity | observations) is updated during the sample period, "
            "then MAINTAINED during the delay because the A-matrix for delay observations is "
            "uninformative (uniform likelihood). Bayesian updating with flat likelihoods preserves "
            "the prior, so beliefs persist automatically. Working memory IS sustained posterior "
            "beliefs under uninformative observations. No special mechanism needed.",
    title="Working Memory: Emergent Network Dynamics vs. Sustained Bayesian Beliefs"
)

### Paradigm 6: Context-Dependent Decision Making

**Origin:** Mante et al. (2013, Nature) recorded from prefrontal cortex while monkeys performed a task with two sensory inputs (motion direction and color) but only one was relevant on each trial, determined by a context cue. This is the "cocktail party problem" of cognitive neuroscience: how does the brain selectively attend to one stream of information while ignoring another?

**Task structure:**
```
  Context cue: MOTION                  Context cue: COLOR
  motion: →→→  color: red               motion: →→→  color: red
  Respond based on motion (→ right)     Respond based on color (→ left if red)
```

**Neuroscience significance:** Mante et al. showed that PFC neurons form a low-dimensional representation where context "rotates" the relevant sensory axis into the decision axis. This is a neural implementation of *attentional gating* — a mechanism that Active Inference models with hierarchical beliefs.

**NeuroGym implementation:** `ContextDecisionMaking-v0` provides two noisy input channels plus a context cue. The agent must respond based on the cued channel while ignoring the other.

In [ ]:
# ── Paradigm 6: Context-Dependent Decision Making ─────────────────────────────

if NEUROGYM_AVAILABLE:
    try:
        env = ngym.make('ContextDecisionMaking-v0')
        obs_list, act_list, rew_list = [], [], []
        ob = env.reset()
        for _ in range(120):
            action = env.action_space.sample()
            ob, reward, done, info = env.step(action)
            obs_list.append(ob)
            act_list.append(action)
            rew_list.append(reward)
            if done:
                break
        observations = np.array(obs_list)
        actions = np.array(act_list)
        rewards = np.array(rew_list)
        plot_task_timecourse(observations, actions, rewards,
                             'Context-Dependent DM: Single Trial')
    except Exception:
        pass

# Synthetic context-dependent decision making
n_steps = 70
obs_ctx = np.zeros((n_steps, 5))  # [fixation, motion_L, motion_R, color_L, color_R]
act_ctx = np.zeros(n_steps, dtype=int)
rew_ctx = np.zeros(n_steps)

# Context: attend to MOTION (represented by an additional input, but here we use the
# trial structure to demonstrate)
context = 'motion'  # This trial: attend to motion
true_motion = 'right'  # Coherent motion is rightward
true_color = 'left'    # Color indicates left

# Fixation: steps 0-9
obs_ctx[0:10, 0] = 1.0

# Stimulus period: steps 10-54
obs_ctx[10:55, 0] = 1.0
# Motion channel: rightward (channel 2 > channel 1)
obs_ctx[10:55, 1] = 0.3 + 0.15 * np.random.randn(45)  # weak left
obs_ctx[10:55, 2] = 0.7 + 0.15 * np.random.randn(45)  # strong right
# Color channel: leftward (channel 3 > channel 4)
obs_ctx[10:55, 3] = 0.7 + 0.15 * np.random.randn(45)  # strong left
obs_ctx[10:55, 4] = 0.3 + 0.15 * np.random.randn(45)  # weak right

# Response period: steps 55-69
obs_ctx[55:70, 0] = 0.0  # fixation off
# Correct response: attend to motion → right → action=2
act_ctx[60:70] = 2
rew_ctx[60] = 1.0

plot_task_timecourse(obs_ctx, act_ctx, rew_ctx,
                     f'Context-Dependent DM: Attend {context.upper()} (Synthetic)',
                     obs_labels=['Fix', 'Motion-L', 'Motion-R', 'Color-L', 'Color-R'])

print(f"Context: attend to {context.upper()}")
print(f"Motion says: {true_motion}  |  Color says: {true_color}")
print(f"Correct response: {true_motion} (following motion context)")
print("If context were COLOR, the correct response would be: left")

In [ ]:
# ── Context DM: RL vs AIF ───────────────────────────────────────────────────────
dual_perspective_box(
    rl_text="An RNN trained with RL learns gating mechanisms that route the context-relevant "
            "sensory stream to the decision output while suppressing the irrelevant stream. "
            "Mante et al. showed this involves rotating the neural state space so that the "
            "relevant axis aligns with the readout direction. The gating is learned implicitly "
            "from reward signals.",
    aif_text="Hierarchical beliefs naturally implement contextual gating. A higher-level belief "
            "about context (motion vs. color) modulates how lower-level sensory evidence is "
            "weighted in the A-matrix likelihood. In context='motion', the A-matrix entries "
            "for motion channels have high precision while color channels have low precision. "
            "Attention IS precision-weighting of prediction errors in the Active Inference framework.",
    title="Context DM: Learned Gating vs. Precision-Weighted Attention"
)

## A1.4 Task Taxonomy: Cognitive Demands

Not all tasks are created equal. Each paradigm places different demands on different cognitive subsystems. The heatmap below organizes our six tasks by five cognitive dimensions:

- **Perception:** How much sensory processing is needed? (random dots >> bandit)
- **Working Memory:** Must information be maintained across a delay? (DMS >> GoNogo)
- **Decision Making:** How complex is the choice? (Two-Step >> GoNogo)
- **Motor Timing:** Does the response require precise timing? (GoNogo >> Bandit)
- **Learning:** Must the agent adapt over trials? (Bandit >> DMS)

This taxonomy helps when choosing tasks for testing specific computational hypotheses. If you want to study exploration, use the Bandit. If you want to study evidence accumulation, use PDM. If you want to study the model-based/model-free distinction, use the Two-Step.

In [ ]:
# ── Task Taxonomy Heatmap ────────────────────────────────────────────────────────
tasks = ['GoNogo', 'Bandit', 'DawTwoStep', 'PDM', 'DMS', 'ContextDM']
demands = ['Perception', 'Working\nMemory', 'Decision\nMaking', 'Motor\nTiming', 'Learning']

# Ratings 0-3: 0=None, 1=Minimal, 2=Moderate, 3=Primary
ratings = np.array([
    [1, 0, 2, 2, 1],  # GoNogo
    [0, 1, 3, 0, 3],  # Bandit
    [0, 1, 3, 0, 3],  # DawTwoStep
    [3, 0, 2, 1, 0],  # PDM
    [2, 3, 1, 0, 0],  # DMS
    [3, 2, 2, 0, 1],  # ContextDM
])

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(ratings, cmap='YlOrRd', aspect='auto', vmin=0, vmax=3)
ax.set_xticks(range(len(demands)))
ax.set_xticklabels(demands, fontsize=10)
ax.set_yticks(range(len(tasks)))
ax.set_yticklabels(tasks, fontsize=10)
for i in range(len(tasks)):
    for j in range(len(demands)):
        ax.text(j, i, str(ratings[i, j]), ha='center', va='center',
                fontsize=12, fontweight='bold',
                color='white' if ratings[i, j] >= 2 else 'black')
ax.set_title('Cognitive Demand Taxonomy', fontsize=13)
plt.colorbar(im, ax=ax, label='Demand Level (0=None, 3=Primary)', shrink=0.8)
plt.tight_layout()
plt.show()

print("Each task emphasizes different cognitive subsystems.")
print("Choose your task based on the computational question you want to ask.")

## A1.5 Mapping Tasks to A/B/C/D Matrices

The core insight of Active Inference is that every cognitive task can be described by a **generative model** with four components:

| Matrix | Name | Encodes | Example (Go/No-Go) |
|--------|------|---------|--------------------|
| **A** | Likelihood | P(observation \| hidden state) | Strong stimulus signals 'go' trial |
| **B** | Transition | P(next state \| current state, action) | Responding in 'go' trial → reward state |
| **C** | Preferences | log P(preferred observations) | Prefer reward, avoid punishment |
| **D** | Prior | P(initial hidden state) | 50/50 go vs. no-go prior |

Let us make this concrete for the Go/No-Go task. The following code constructs an explicit A-matrix mapping hidden states to observations. This is a *conceptual* demonstration — the full generative model with B, C, and D matrices would be implemented in a dedicated Active Inference solver (see Appendix A4).

In [ ]:
# ── GoNogo as a Generative Model ────────────────────────────────────────────────
# This is a conceptual demonstration — full implementation in Appendix A4

state_names = ['go_pre', 'go_stim', 'go_resp', 'nogo_pre', 'nogo_stim', 'nogo_resp']
obs_names = ['null', 'weak', 'strong', 'reward', 'no_reward']
action_names = ['wait', 'respond', 'withhold']

# A matrix: P(observation | hidden state)
A = np.zeros((5, 6))
A[0, 0] = 1.0; A[0, 3] = 1.0      # pre-stimulus → null observation
A[2, 1] = 0.9; A[1, 1] = 0.1      # go_stim → strong (mostly)
A[1, 4] = 0.9; A[2, 4] = 0.1      # nogo_stim → weak (mostly)
A[3, 2] = 1.0                       # go_resp (if responded) → reward
A[4, 5] = 1.0                       # nogo_resp (if responded) → no_reward

fig, ax = plt.subplots(figsize=(8, 5))
plot_matrix_heatmap(A, title='A Matrix: P(observation | state)',
                    row_labels=obs_names, col_labels=state_names, ax=ax)
plt.tight_layout()
plt.show()

print("The A matrix encodes the task's observation structure.")
print("Strong stimuli signal 'go' trials; weak stimuli signal 'no-go'.")
print("See Appendix A4 for the full generative model with B, C, D matrices.")

In [ ]:
# ── A/B/C/D Mapping: RL vs AIF ─────────────────────────────────────────────────
dual_perspective_box(
    rl_text="In RL, the task structure is implicit in the reward function and transition dynamics. "
            "The agent does not have an explicit 'A matrix' — instead, it learns a mapping from "
            "observations to values (or actions) through trial and error. The observation structure "
            "is handled by the neural network architecture (CNN for pixels, MLP for vectors). "
            "Task knowledge is distributed across network weights.",
    aif_text="In Active Inference, the A/B/C/D matrices ARE the agent's knowledge of the task. "
            "The A-matrix is the observation model (what do I expect to see given each state?). "
            "The B-matrix is the transition model (how do my actions change the world?). "
            "The C-vector is preferences (what outcomes do I want?). "
            "The D-vector is prior beliefs (what state do I start in?). "
            "Everything is explicit and inspectable.",
    title="Task Representation: Implicit Network Weights vs. Explicit Generative Model"
)

## A1.6 Exercises

### Exercise 1 (Guided): Reaction Time Analysis

The ReadySetGo task requires precise motor timing: the agent observes a "ready" and "set" signal separated by an interval, then must produce a "go" response after the same interval. Run the task below (or use the synthetic data) and plot the distribution of response times.

**Goal:** Understand how temporal structure appears in the Gymnasium API.

In [ ]:
# ── Exercise 1: Reaction Time Analysis ─────────────────────────────────────────

if NEUROGYM_AVAILABLE:
    try:
        env = ngym.make('ReadySetGo-v0')
        response_times = []
        for trial in range(50):
            ob = env.reset()
            for t in range(200):
                action = 1 if t > 40 else 0  # Simple fixed-delay policy
                ob, reward, done, info = env.step(action)
                if done:
                    if reward > 0:
                        response_times.append(t)
                    break
        rt = np.array(response_times) if response_times else None
    except Exception:
        rt = None
else:
    rt = None

if rt is None:
    # Synthetic reaction time data: ex-Gaussian distribution (typical of human RTs)
    mu, sigma, tau = 350, 30, 50  # ms
    gaussian_part = np.random.normal(mu, sigma, 200)
    exponential_part = np.random.exponential(tau, 200)
    rt = gaussian_part + exponential_part
    rt = rt[rt > 200]  # Remove impossibly fast responses

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(rt, bins=30, color=COLORS['rl'], alpha=0.7, edgecolor='white')
axes[0].axvline(np.median(rt), color=COLORS['highlight'], linewidth=2,
                linestyle='--', label=f'Median: {np.median(rt):.0f}')
axes[0].set_xlabel('Response Time')
axes[0].set_ylabel('Count')
axes[0].set_title('Response Time Distribution')
axes[0].legend()

# QQ-style: sorted RTs
axes[1].plot(np.sort(rt), np.linspace(0, 1, len(rt)), color=COLORS['aif'], linewidth=2)
axes[1].set_xlabel('Response Time')
axes[1].set_ylabel('Cumulative Probability')
axes[1].set_title('CDF of Response Times')

plt.tight_layout()
plt.show()

print(f"Mean RT: {np.mean(rt):.1f} | Median RT: {np.median(rt):.1f} | Std: {np.std(rt):.1f}")
print("Note the rightward skew — this is characteristic of the ex-Gaussian distribution")
print("commonly observed in human reaction time data.")

### Exercise 2 (Intermediate): Psychometric Curve for PDM

Generate a psychometric curve for the Perceptual Decision Making task by running many trials at different coherence levels and computing accuracy at each level.

**Goal:** Reproduce the classic sigmoid relationship between stimulus strength and choice accuracy.

In [ ]:
# ── Exercise 2: Psychometric Curve ─────────────────────────────────────────────

# Simulate a Bayesian ideal observer at different coherence levels
coherence_levels = np.array([0.02, 0.05, 0.10, 0.20, 0.35, 0.50, 0.70, 0.90])
n_trials_per = 100
n_timesteps = 50
noise_std = 0.5

accuracy = np.zeros(len(coherence_levels))
mean_rt = np.zeros(len(coherence_levels))

for ci, coh in enumerate(coherence_levels):
    correct = 0
    rts = []
    for trial in range(n_trials_per):
        true_dir = np.random.choice([-1, 1])
        # Accumulate evidence
        evidence = np.cumsum(coh * true_dir + noise_std * np.random.randn(n_timesteps))
        # Decision at threshold or end of trial
        threshold = 3.0
        crossed = np.where(np.abs(evidence) > threshold)[0]
        if len(crossed) > 0:
            rt_step = crossed[0]
            decision = np.sign(evidence[rt_step])
        else:
            rt_step = n_timesteps - 1
            decision = np.sign(evidence[-1])
        correct += (decision == true_dir)
        rts.append(rt_step)
    accuracy[ci] = correct / n_trials_per
    mean_rt[ci] = np.mean(rts)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: psychometric curve
axes[0].plot(coherence_levels * 100, accuracy, 'o-', color=COLORS['rl'],
             linewidth=2, markersize=8)
axes[0].axhline(y=0.5, color='gray', linestyle=':', linewidth=0.5)
axes[0].set_xlabel('Coherence (%)')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Psychometric Curve (Ideal Observer)')
axes[0].set_ylim(0.4, 1.05)

# Right: chronometric curve (RT vs coherence)
axes[1].plot(coherence_levels * 100, mean_rt, 's-', color=COLORS['aif'],
             linewidth=2, markersize=8)
axes[1].set_xlabel('Coherence (%)')
axes[1].set_ylabel('Mean Decision Time (steps)')
axes[1].set_title('Chronometric Curve (Speed-Accuracy Tradeoff)')

plt.tight_layout()
plt.show()

print("The psychometric curve shows the classic sigmoid shape: accuracy improves with coherence.")
print("The chronometric curve shows the speed-accuracy tradeoff: easier trials are decided faster.")
print()
print("TRY: Change the threshold (line 'threshold = 3.0') and re-run.")
print("  Higher threshold -> slower but more accurate (conservative)")
print("  Lower threshold  -> faster but less accurate (liberal)")

### Exercise 3 (Open-ended): Design A/B/C/D for a Novel Task

Choose a cognitive task NOT in the six above (e.g., Wisconsin Card Sorting, Stroop, Reversal Learning, or invent your own) and design the A/B/C/D matrices for it.

**Steps:**
1. Define the hidden states (what is the agent uncertain about?)
2. Define the observations (what does the agent see?)
3. Define the actions (what can the agent do?)
4. Write out the A-matrix: P(observation | hidden state)
5. Write out the B-matrix: P(next state | current state, action)
6. Write out the C-vector: which observations are preferred?
7. Write out the D-vector: prior beliefs over initial states

**Starter code below** — fill in the matrices for your chosen task:

In [ ]:
# ── Exercise 3: Design Your Own Generative Model ───────────────────────────────

# === YOUR TASK: [Name your task here] ===

# Step 1: Define state, observation, and action spaces
my_state_names = ['state_0', 'state_1', 'state_2']   # TODO: replace
my_obs_names = ['obs_0', 'obs_1', 'obs_2']           # TODO: replace
my_action_names = ['action_0', 'action_1']            # TODO: replace

n_states = len(my_state_names)
n_obs = len(my_obs_names)
n_actions = len(my_action_names)

# Step 2: A-matrix — P(observation | hidden state)
my_A = np.zeros((n_obs, n_states))
# TODO: Fill in your A-matrix
# Example: my_A[0, 0] = 0.9  # state_0 usually produces obs_0
# Make sure columns sum to 1!
my_A[:, :] = 1.0 / n_obs  # Placeholder: uniform (replace me!)

# Step 3: B-matrix — P(next state | current state, action)
my_B = np.zeros((n_actions, n_states, n_states))
# TODO: Fill in your B-matrices
for a in range(n_actions):
    my_B[a] = np.eye(n_states)  # Placeholder: identity (replace me!)

# Step 4: C-vector — log preferences over observations
my_C = np.zeros(n_obs)
# TODO: Set preferences (positive = preferred, negative = aversive)

# Step 5: D-vector — prior over initial states
my_D = np.ones(n_states) / n_states  # Uniform prior
# TODO: Adjust if some initial states are more likely

# Visualize your A-matrix
fig, ax = plt.subplots(figsize=(7, 4))
plot_matrix_heatmap(my_A, title='Your A Matrix: P(obs | state)',
                    row_labels=my_obs_names, col_labels=my_state_names, ax=ax)
plt.tight_layout()
plt.show()

# Validation
col_sums = my_A.sum(axis=0)
if np.allclose(col_sums, 1.0):
    print("A-matrix columns sum to 1.0 — valid probability distribution.")
else:
    print(f"WARNING: A-matrix columns sum to {col_sums} (should be 1.0 each).")
    print("Each column must be a valid probability distribution over observations.")

## Further Reading

- **NeuroGym GitHub:** [github.com/neurogym/neurogym](https://github.com/neurogym/neurogym) — source code, task documentation, and examples.
- **Yang, G.R. & Wang, X.J. (2020).** Artificial neural networks as models of neural information processing. *Annual Review of Neuroscience*, 43, 509-536. — The paper introducing NeuroGym's task-training framework.
- **Ehrlich, D.B., Stone, J.T., Brandfonbrener, D., Atanasov, A., & Murray, J.D. (2021).** PsychRNN: An accessible and flexible Python package for training recurrent neural network models on cognitive tasks. *eNeuro*, 8(1). — The predecessor to NeuroGym; TensorFlow 1.x only.
- **Daw, N.D., Gershman, S.J., Seymour, B., Dayan, P., & Dolan, R.J. (2011).** Model-based influences on humans' choices and striatal prediction errors. *Neuron*, 69(6), 1204-1215. — The two-step task paper.
- **Mante, V., Sussillo, D., Shenoy, K.V., & Newsome, W.T. (2013).** Context-dependent computation by recurrent dynamics in prefrontal cortex. *Nature*, 503(7474), 78-84. — Context-dependent decision making.
- **Gold, J.I. & Shadlen, M.N. (2007).** The neural basis of decision making. *Annual Review of Neuroscience*, 30, 535-574. — Comprehensive review of evidence accumulation.
- **Ji-An, Li, Benna, M.K., & Bhatt, N.S., & Bhatt, U.S., & Mattar, M.G. (2025).** Tiny RNNs for cognitive neuroscience. *Nature*. — Modern approach to training small, interpretable RNNs on cognitive tasks.

---

*Next: Appendix A2 goes deeper into the Daw Two-Step task, building both model-free and model-based RL agents and comparing them to an Active Inference agent.*